In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')

In [15]:
#load the cleaned dataset
df = pd.read_csv('Data\Processed\cleaned_data.csv')

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 779425 entries, 0 to 779424
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      779425 non-null  int64  
 1   StockCode    779425 non-null  str    
 2   Description  779425 non-null  str    
 3   Quantity     779425 non-null  int64  
 4   InvoiceDate  779425 non-null  str    
 5   Price        779425 non-null  float64
 6   Customer ID  779425 non-null  float64
 7   Country      779425 non-null  str    
 8   Sales        779425 non-null  float64
 9   Month        779425 non-null  str    
dtypes: float64(3), int64(2), str(5)
memory usage: 59.5 MB


In [ ]:
#change data type of invoice date from str to datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
print(df['InvoiceDate'].dtype)

datetime64[us]


In [22]:
#Create Snapshot Date
#This is used for Recency calculation.

snapshot_date = (df["InvoiceDate"].max() + pd.Timedelta(days=1))

print(snapshot_date)

2011-12-10 12:50:00


In [24]:
df.shape

(779425, 10)

In [27]:
# RMF table creation
rfm = df.groupby("Customer ID").agg({
    "InvoiceDate":lambda x:(
            snapshot_date - x.max()
        ).days, 
        "Invoice": "nunique",
        "Sales": "sum"
})

rfm.columns = [
    "Recency",
    "Frequency",
    "Monetary"
]

rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326,12,77556.46
12347.0,2,8,4921.53
12348.0,75,5,2019.40
12349.0,19,4,4428.69
12350.0,310,1,334.40


In [30]:
rfm.shape

(5878, 3)

In [31]:
rfm.describe()

,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000
mean,201.331916,6.289384,2955.904095
std,209.338707,13.009406,14440.852688
min,1.000000,1.000000,2.950000
25%,26.000000,1.000000,342.280000
50%,96.000000,3.000000,867.740000
75%,380.000000,7.000000,2248.305000
max,739.000000,398.000000,580987.040000


This means:

779,425 transactions
5,878 unique customers

This is sufficient for meaningful customer segmentation.

In [ ]:
#save rfm table
rfm.to_csv('Data/Processed/rfm_table.csv')